## Phase 3: Unleashing the Model (XGBoost)
Because our signals are so subtle and "nonlinear" (meaning they don't follow a straight line), a simple linear model would fail. XGBoost is perfect here because it is a "Decision Tree" model that can find complex patterns—like "if temp is high AND odometer is high AND vibration is shaky, then alert."

**Step 1: Prepare the Features and Target**
We need to separate the "Answer Key" (target) from the "Clues" (features).

In [1]:
import pandas as pd
import numpy as np

In [2]:
# read the data from raw
df_master= pd.read_csv('../data/raw/master.csv')

In [3]:
df_master

,date,asset_id,odometer,ambient_temp,coolant_temp,oil_pressure,engine_load,vibration_index,daily_utilization,days_since_service,...,target,next_failure_date,days_to_failure,temp_delta,oil_press_std_7d,ambient_temp_mean_7d,load_7day_std,thermal_stress,stress_7day_avg,vibration_7day_std
0,0,ASSET_000,26083,60.626323,178.162306,39.945186,53.998997,12.141658,0.0,24,...,0.0,NaN,NaN,117.535983,NaN,NaN,NaN,3273.760605,NaN,NaN
1,1,ASSET_000,26313,69.366307,184.745893,47.143099,57.272999,10.542560,230.0,25,...,0.0,NaN,NaN,115.379586,NaN,NaN,NaN,3972.816431,NaN,NaN
2,2,ASSET_000,26648,75.266662,194.447326,48.744780,65.994439,9.572207,335.0,26,...,0.0,NaN,NaN,119.180664,NaN,NaN,NaN,4967.181119,NaN,NaN
3,3,ASSET_000,26802,74.706538,204.270394,47.374386,84.397616,10.355551,154.0,27,...,0.0,NaN,NaN,129.563856,NaN,NaN,NaN,6305.053716,NaN,NaN
4,4,ASSET_000,26990,67.792336,186.983559,44.892239,77.369321,9.708306,188.0,28,...,0.0,NaN,NaN,119.191223,NaN,NaN,NaN,5245.046988,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29995,295,ASSET_099,98147,99.173425,221.340877,48.744923,55.940903,11.510003,324.0,364,...,0.0,NaN,NaN,122.167452,8.027525,75.196806,13.922727,5547.850917,5148.472897,2.232690
29996,296,ASSET_099,98463,77.501352,189.813959,43.100325,69.410565,10.850131,316.0,365,...,0.0,NaN,NaN,112.312606,7.363759,77.735778,13.915067,5379.412690,5339.381637,2.316976
29997,297,ASSET_099,98635,75.341896,194.096033,56.561654,53.027089,12.302767,172.0,366,...,0.0,NaN,NaN,118.754137,6.355345,78.826900,11.698628,3995.161388,5046.599715,2.271408
29998,298,ASSET_099,98766,97.753733,218.017023,40.010038,61.236116,15.282265,131.0,367,...,0.0,NaN,NaN,120.263290,6.942517,80.561339,10.690936,5986.058906,4988.611155,2.464413


In [ ]:
# Select the best features/predictors we've found
# Define final feature set based on objectives and heatmap
model_features = [
    # Metadata
    'asset_type', 'purchase_year', 
    
    # Usage Metrics
    'odometer', 'utilization_7day_avg', 'days_since_service',
    
    # Historical Reliability (Dropping downtime as it's perfectly collinear with count)
    'historical_failure_count',
    
    # Telemetry Trends (The engineered sensors)
    'vibration_index', 'vibration_7day_std', 
    'temp_delta', 'oil_press_std_7d', 
    'load_7day_std', 'stress_7day_avg'
]

In [5]:
df_master[model_features]

,asset_type,purchase_year,odometer,utilization_7day_avg,days_since_service,historical_failure_count,vibration_index,vibration_7day_std,temp_delta,oil_press_std_7d,load_7day_std,stress_7day_avg
0,Light Duty,2021,26083,NaN,24,0,12.141658,NaN,117.535983,NaN,NaN,NaN
1,Light Duty,2021,26313,NaN,25,0,10.542560,NaN,115.379586,NaN,NaN,NaN
2,Light Duty,2021,26648,NaN,26,0,9.572207,NaN,119.180664,NaN,NaN,NaN
3,Light Duty,2021,26802,NaN,27,0,10.355551,NaN,129.563856,NaN,NaN,NaN
4,Light Duty,2021,26990,NaN,28,0,9.708306,NaN,119.191223,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
29995,Heavy Duty,2017,98147,242.142857,364,0,11.510003,2.232690,122.167452,8.027525,13.922727,5148.472897
29996,Heavy Duty,2017,98463,246.142857,365,0,10.850131,2.316976,112.312606,7.363759,13.915067,5339.381637
29997,Heavy Duty,2017,98635,244.000000,366,0,12.302767,2.271408,118.754137,6.355345,11.698628,5046.599715
29998,Heavy Duty,2017,98766,241.714286,367,0,15.282265,2.464413,120.263290,6.942517,10.690936,4988.611155


## 5. Temporal Split

There are two primary ways to split time-series data, and they serve different "Data Science" purposes.

Given our project objective—building a robust classification pipeline—we need to choose the one that proves our model works in a real-world fleet environment.

1. Temporal Split

This is best for answering: "If I train on last year's data, can I predict next month's failures?"

Pros: It perfectly mimics real-time deployment. It handles "Seasonality" or "Global Trends" (like the simulation getting more "shaky" after Day 200).

Cons: If the data have the same Asset_001 in both the "Past" and the "Future," the model might just learn that Asset_001 is a "bad truck" rather than learning the symptoms of a bad truck.

2. Asset-Based Split 

This is best for answering: "If I buy a new truck tomorrow, can my model predict its failures?"

Pros: It proves the model is learning mechanical physics (heat/vibration) rather than just memorizing specific vehicle IDs.

The "Gold Standard" Solution: The Combined Split

In professional BI and Data Science, the most robust way is to combine both. We use the Temporal Split (Days 0–240) but we also ensure we handle the Categorical Encoding and Scaling within that timeframe.

In [ ]:
# 1. Define the cutoff for a 80/20 time split
cutoff_day = 240 # Train model on this time line data
end_day = 270 # Test model on (240 -270) time line data

# 2. Filter the Data
train_df = df_master[df_master['date'] < cutoff_day].copy()
test_df = df_master[(df_master['date'] >= cutoff_day) & (df_master['date'] <= end_day)].copy()

# 3. Encode & Scale (Crucial Step)
# We must apply encoding/scaling AFTER the split to prevent "Look-ahead" bias
X_train = pd.get_dummies(train_df[model_features], columns=['asset_type'], drop_first=True)
X_test = pd.get_dummies(test_df[model_features], columns=['asset_type'], drop_first=True)

# 4. Handle NaNs and Scaling
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

# List of numeric columns from model_features
numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()

X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols].fillna(X_train[numeric_cols].median()))
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols].fillna(X_train[numeric_cols].median()))

y_train = train_df['target']
y_test = test_df['target']

print(f"Training on Past (Days 0-239): {X_train.shape}")
print(f"Testing on Future (Days 240-270): {X_test.shape}")

Buffer Logic: correctly identified the "Censored Data" problem. By dropping the last 30 days, it is ensure that every "0" in test set is a true zero, not just a vehicle that was about to fail on day 305.

Data Integrity: By using the median of the training set to fill NaNs in the test set, it's ensure that model doesn't "peek" into the future statistics of the fleet.

In [ ]:
# saving data in the data/processed
import os 
os.makedirs('../data/processed', exist_ok=True)
#df_master.to_csv('../data/raw/master.csv', index=False)

X_train.to_csv('../data/processed/X_train_temporal.csv', index=False)
X_test.to_csv('../data/processed/X_test_temporal.csv', index=False)
y_train.to_csv('../data/processed/y_train_temporal.csv', index=False)
y_test.to_csv('../data/processed/y_test_temporal.csv', index=False)

3. Why balancing is necessary (The "Accuracy Trap")

If don't use weights or balancing, model will fall into the Accuracy Trap.
Since only ~3% of data are failures, a model can achieve 97% accuracy simply by predicting that "Nothing will ever break." That model is 97% accurate but 0% useful.

By using weights, it force the model to prioritize Recall—which aligns with project objective of catching failures before they happen.

Since we are comparing Random Forest, XGBoost, and LightGBM, we have a major advantage: these models are designed with "class weighting" parameters that often outperform SMOTE in industrial predictive maintenance.

Here is the breakdown of whether should balance the data and the most effective way to do it for fleet project.

1. Should use SMOTE?

    Verdict: Not recommended for this specific project.

    While SMOTE is great for generic datasets, it has specific drawbacks for sensor/fleet data:

    Physics Violation: SMOTE creates "synthetic" rows by averaging two real rows. In predictive maintenance, this might create a record with high vibration but low temperature that doesn't actually follow the laws of thermodynamics, confusing model.

    Temporal Leakage: Since here used a Temporal Split, SMOTE might accidentally create synthetic data points that "blur" the line between  training past and testing future.

    Overfitting: SMOTE can make the model too confident about the synthetic patterns, leading to a high "training" score but a failure to catch real breakdowns in the field.

2. The Better Alternative: Cost-Sensitive Learning

    Instead of changing the data (SMOTE), change the importance the model gives to each row. This is called Cost-Sensitive Learning.

    All three models chosen have a built-in "Weighting" parameter:

**table**

| Model | Parameter | How it works |
| :--: | :--: | :-- |
| Random Forest | class_weight='balanced' | Automatically gives more "voting power" to the rare failure cases. |
| XGBoost | scale_pos_weight | Penalizes the model more heavily for missing a failure than for a false alarm. |
| LightGBM | is_unbalance=True | Optimizes the training specifically for datasets where one class is very rare. |

Why these three?

**Random Forest:**  Excellent at capturing non-linear relationships without much tuning. Because it uses "Bagging" (averaging trees), it is less likely to overfit to a single noisy sensor.

**XGBoost:**  The gold standard for tabular data. Its "Boosting" method focuses specifically on the "hard-to-predict" cases (the failures) in each round of training.

**LightGBM:**  Specifically designed for speed and large datasets. In a real-world fleet scenario with millions of pings per hour, LightGBM is often the production choice because it uses Gradient-based One-Side Sampling (GOSS) to find patterns faster.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import f1_score, recall_score, precision_score

# Calculate class weight for imbalance
pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

# Define the models
models = {
    "Random Forest": RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42),
    
    "XGBoost": XGBClassifier(n_estimators=100, scale_pos_weight=pos_weight, learning_rate=0.05, eval_metric='logloss', random_state=42),
    
    "LightGBM": LGBMClassifier(n_estimators=100, is_unbalance=True, learning_rate=0.05, random_state=42)
}

# Loop through and evaluate
results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1-Score": f1_score(y_test, y_pred)
    })

# Display Results
import pandas as pd
results_df = pd.DataFrame(results).sort_values(by="F1-Score", ascending=False)
print(results_df)

In [ ]:
# Calculate the exact ratio of Healthy to Failure cases
ratio = (y_train == 0).sum() / (y_train == 1).sum()

# 1. Random Forest with Balanced Weights
rf_model = RandomForestClassifier(class_weight='balanced', n_estimators=100)

# 2. XGBoost with scale_pos_weight
xgb_model = XGBClassifier(scale_pos_weight=ratio, n_estimators=100)

# 3. LightGBM with is_unbalance
lgbm_model = LGBMClassifier(is_unbalance=True, n_estimators=100)

What to look for in results

High Recall: This model is "Safety First" model—it catches almost every breakdown but might have some false alarms.

High Precision: This model is "Efficiency" model—it only alerts when it is very sure, meaning mechanics don't waste time on healthy trucks.

F1-Score: This is the "Goldilocks" metric that balances both.

--------------------------------------------------------------------------------------------------------------------

In [ ]:
# 1. Define the cutoff for a 80/20 time split
cutoff_day = 240
end_day = 270 

# 2. Filter the Data
train_df = df_master[df_master['date'] < cutoff_day].copy()
test_df = df_master[(df_master['date'] >= cutoff_day) & (df_master['date'] <= end_day)].copy()

# 3. Encode & Scale (Crucial Step)
# We must apply encoding/scaling AFTER the split to prevent "Look-ahead" bias
X_train = pd.get_dummies(train_df[model_features], columns=['asset_type'], drop_first=True)
X_test = pd.get_dummies(test_df[model_features], columns=['asset_type'], drop_first=True)

# 4. Handle NaNs and Scaling
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

# List of numeric columns from your model_features
numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()

X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols].fillna(X_train[numeric_cols].median()))
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols].fillna(X_train[numeric_cols].median()))

y_train = train_df['target']
y_test = test_df['target']

print(f"Training on Past (Days 0-239): {X_train.shape}")
print(f"Testing on Future (Days 240-270): {X_test.shape}")